# SigLIP Diagnostic & Saliency-Plateau Benchmark
This notebook investigates SigLIP inference, query templates, and probability calibration.

### Objectives:
1. **SigLIP2 Notebook Parity**: Test prompt templates from `MSSearch/vis/SigLIP2_inference.ipynb` (`The photo of {label}.` vs `a photo of {q}` vs raw tags).
2. **Calibration Diagnosis**: Compare raw cosine similarity $\cos(I, T)$, SigLIP native sigmoid $\sigma(\text{logit})$, and linear dynamic range calibration $[0.02, 0.24] \to [0.0, 1.0]$.
3. **Stage 3 Saliency Plateau**: Measure $\Delta S = S_{\text{max\_child}} - S_{\text{parent}}$ on parent vs 4 child quadtree boxes for target vs background to determine why $\tau = 0.04$ stops early with sigmoid probabilities.

In [ ]:
# ==============================================================================
# CELL 1: SETUP & REPOSITORY SYNC
# ==============================================================================
import os
import sys
import subprocess
import torch

TARGET_BRANCH = 'draft/improve_all'
REPO_DIR = '/kaggle/working/MSSearch' if os.path.exists('/kaggle') else os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(REPO_DIR, 'mssearch')) and os.path.exists('mssearch'):
    REPO_DIR = os.path.abspath(os.getcwd())

if os.path.exists(REPO_DIR):
    print(f'[SYNC] Pulling latest {TARGET_BRANCH} in {REPO_DIR}...')
    subprocess.run(['git', 'fetch', 'origin', TARGET_BRANCH], cwd=REPO_DIR)
    subprocess.run(['git', 'checkout', TARGET_BRANCH], cwd=REPO_DIR)
    subprocess.run(['git', 'pull', 'origin', TARGET_BRANCH], cwd=REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'[ENVIRONMENT] Device: {device} | PyTorch: {torch.__version__}')


In [ ]:
# ==============================================================================
# CELL 2: RUN COMPREHENSIVE SIGLIP DIAGNOSTIC
# Evaluates Synthetic crops and HRBench Sample 0 crops side-by-side
# ==============================================================================
import sys
# Import and run diagnostic module
from pilot.multisource_hrbench.debug_siglip import main

sys.argv = ['debug_siglip.py', '--sample_idx', '0']
main()
